# MMCAD Render Encoder: BRep-Anchored SigLIP Training

Trains a **SigLIP-base** render encoder to map photorealistic synthetic images into the
BRep embedding space from the v4 tri-modal model. Uses the same anchor-alignment approach
as the sketch encoder, with cosine + InfoNCE loss at Matryoshka scales.

**Architecture**: SigLIP vision encoder (mean-pool) -> 2-layer projection head -> 768-dim shared space

**Key differences from sketch encoder**:
- SigLIP vision backbone instead of ViT (sigmoid-based contrastive pretraining, better for retrieval)
- SigLIP normalization (mean=0.5, std=0.5) instead of ImageNet normalization
- Color jitter augmentation (leverages photorealistic image properties)
- Flat dataset: `{uid}_synthetic.png` per model
- No dual-view augmentation (single render per UID)

In [ ]:
# Cell 1: Environment Setup
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q transformers accelerate
    print('Colab environment ready.')
else:
    print('Local environment.')

In [ ]:
# Cell 2: Extract synthetic render images
import os, zipfile, time

if IS_COLAB:
    DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
    EXTRACT_DIR = '/content/mmcad_data/synthetic_renders'
    ZIP_PATH = f'{DRIVE_DIR}/synthetic_renders.zip'
else:
    DRIVE_DIR = r'C:\Users\anush\Desktop\MMCAD'
    EXTRACT_DIR = os.path.join(DRIVE_DIR, 'synthetic_renders')
    ZIP_PATH = os.path.join(DRIVE_DIR, 'synthetic_renders.zip')

if os.path.exists(EXTRACT_DIR) and len(os.listdir(EXTRACT_DIR)) > 100:
    n = len([f for f in os.listdir(EXTRACT_DIR) if f.endswith('.png')])
    print(f'Renders already extracted: {n:,} images in {EXTRACT_DIR}')
else:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print(f'Extracting renders from {ZIP_PATH}...')
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    elapsed = time.time() - t0
    # Handle case where zip extracts into a subdirectory
    subdirs = [d for d in os.listdir(EXTRACT_DIR)
               if os.path.isdir(os.path.join(EXTRACT_DIR, d))]
    if len(subdirs) == 1 and not any(
            f.endswith('.png') for f in os.listdir(EXTRACT_DIR)):
        EXTRACT_DIR = os.path.join(EXTRACT_DIR, subdirs[0])
        print(f'  -> Found nested directory, using: {EXTRACT_DIR}')
    n = len([f for f in os.listdir(EXTRACT_DIR) if f.endswith('.png')])
    print(f'Extracted {n:,} images in {elapsed:.0f}s')

RENDER_ROOT = EXTRACT_DIR
print(f'Render root: {RENDER_ROOT}')

In [ ]:
# Cell 3: Imports
import os, gc, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} ({p.total_memory / 1024**3:.1f} GB)')
else:
    print('No GPU available')

In [ ]:
# Cell 4: Configuration
class Config:
    # === PATHS ===
    if IS_COLAB:
        DATA_ROOT = '/content/mmcad_data'
        DRIVE_DIR = '/content/drive/MyDrive/MMCAD'
        ANCHOR_DIR = f'{DRIVE_DIR}/sketch_anchors'   # Reuse anchors from sketch encoder
        SAVE_DIR = f'{DRIVE_DIR}/render_encoder_v1'
        LOCAL_SAVE_DIR = '/content/render_v1_ckpts'
    else:
        DATA_ROOT = r'C:\Users\anush\Desktop\MMCAD'
        ANCHOR_DIR = os.path.join(DATA_ROOT, 'sketch_anchors')
        SAVE_DIR = os.path.join(DATA_ROOT, 'render_encoder_v1')
        LOCAL_SAVE_DIR = SAVE_DIR

    # === SHARED SPACE (matches v4 exactly) ===
    D_SHARED = 768
    MATRYOSHKA_DIMS = [128, 256, 512, 768]
    MATRYOSHKA_WEIGHTS = [1.0, 1.0, 1.0, 1.5]

    # === SIGLIP ENCODER CONFIG ===
    SIGLIP_MODEL = 'google/siglip-base-patch16-224'
    IMAGE_SIZE = 224

    # === TRAINING ===
    BATCH_SIZE = 256
    BACKBONE_LR = 1e-5        # SigLIP backbone (pretrained, lower LR)
    PROJ_LR = 1e-4            # projection head (new, higher LR)
    WEIGHT_DECAY = 0.01
    NUM_EPOCHS = 15
    WARMUP_EPOCHS = 2
    TEMPERATURE = 0.05        # Matches v4 final tau
    GRAD_CLIP = 1.0

    # === MISC ===
    NUM_WORKERS = 2
    SEED = 42
    K_VALUES = [1, 5, 10]
    SAVE_EVERY = 5

config = Config()
os.makedirs(config.SAVE_DIR, exist_ok=True)
os.makedirs(config.LOCAL_SAVE_DIR, exist_ok=True)
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {device}')
print(f'Anchor dir: {config.ANCHOR_DIR}')
print(f'Save dir:   {config.SAVE_DIR}')
print(f'SigLIP: {config.SIGLIP_MODEL}')
print(f'Temperature: {config.TEMPERATURE}')
print(f'Batch size: {config.BATCH_SIZE}')

---
## Phase 1: BRep Anchors (Reuse from Sketch Encoder)

The BRep anchors and train/val split were pre-computed during sketch encoder training
and saved to `sketch_anchors/`. This notebook **reuses** those anchors — both modalities
(sketch and render) align to the same BRep embedding space.

If anchors don't exist, run the sketch encoder notebook (`mmcad_sketch_encoder`) Phase 1 first.

In [ ]:
# Cell 5: Verify & Load BRep Anchors
BREP_ANCHOR_PATH = os.path.join(config.ANCHOR_DIR, 'brep_anchors.pt')
SPLIT_PATH = os.path.join(config.ANCHOR_DIR, 'split_info.pt')

assert os.path.exists(BREP_ANCHOR_PATH), (
    f'BRep anchors not found at {BREP_ANCHOR_PATH}.\n'
    'Run sketch encoder notebook Phase 1 first to compute anchors.')
assert os.path.exists(SPLIT_PATH), (
    f'Split info not found at {SPLIT_PATH}.\n'
    'Run sketch encoder notebook Phase 1 first to compute split.')

brep_anchors = torch.load(BREP_ANCHOR_PATH, map_location='cpu', weights_only=True)
split_info = torch.load(SPLIT_PATH, map_location='cpu', weights_only=True)

print(f'Loaded {len(brep_anchors):,} BRep anchors')
print(f'Anchor dim: {next(iter(brep_anchors.values())).shape}')
print(f'Split: {len(split_info["train_uids"]):,} train, {len(split_info["val_uids"]):,} val')

---
## Phase 2: Train SigLIP Render Encoder

Loads pre-computed BRep anchors and trains a SigLIP-base encoder to match
the BRep embedding space. Identical loss function and Matryoshka structure
as the sketch encoder — only the vision backbone and image transforms differ.

In [ ]:
# Cell 6: SigLIP Render Encoder
from transformers import SiglipVisionModel

class SigLIPRenderEncoder(nn.Module):
    """SigLIP vision encoder + 2-layer projection head -> 768-dim shared space.

    Uses SigLIP's pooled patch representations (no CLS token, unlike ViT).
    Projection head mirrors ViTSketchEncoder: Linear -> LN -> GELU -> Drop -> Linear.
    """
    def __init__(self, model_name='google/siglip-base-patch16-224', d_out=768):
        super().__init__()
        self.vision = SiglipVisionModel.from_pretrained(model_name)
        h = self.vision.config.hidden_size   # 768 for siglip-base
        self.projection = nn.Sequential(
            nn.Linear(h, d_out),
            nn.LayerNorm(d_out),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_out, d_out),
        )

    def forward(self, pixel_values):
        # SigLIP pools all patch tokens (no CLS token)
        pooled = self.vision(pixel_values=pixel_values).pooler_output
        return self.projection(pooled)

# Quick param count
_tmp = SigLIPRenderEncoder(config.SIGLIP_MODEL, config.D_SHARED)
n_params = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f'SigLIPRenderEncoder: {n_params:.1f}M params')
del _tmp

In [ ]:
# Cell 7: Render Dataset + Transforms

# SigLIP normalization: maps [0,1] -> [-1,1]
SIGLIP_MEAN = [0.5, 0.5, 0.5]
SIGLIP_STD  = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(config.IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=SIGLIP_MEAN, std=SIGLIP_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(config.IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=SIGLIP_MEAN, std=SIGLIP_STD),
])


class RenderAnchorDataset(Dataset):
    """Loads photorealistic render images + pre-computed BRep anchors.

    Images are named {uid}_synthetic.png in a flat directory.
    Skips corrupt images gracefully with random replacement.
    """
    def __init__(self, uids, render_root, brep_anchors, transform):
        self.transform = transform
        self.render_root = render_root
        self.brep_anchors = brep_anchors
        self._bad_paths = set()

        # Filter: must have render file + anchor
        self.samples = []
        for uid in uids:
            uid_str = str(uid).strip()
            img_path = os.path.join(render_root, f'{uid_str}_synthetic.png')
            if os.path.exists(img_path) and uid_str in brep_anchors:
                self.samples.append((uid_str, img_path))

        print(f'RenderAnchorDataset: {len(self.samples):,} samples '
              f'(from {len(uids):,} UIDs, render_root={render_root})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        for attempt in range(10):
            try:
                return self._load_sample(idx)
            except Exception:
                uid_str, img_path = self.samples[idx]
                if img_path not in self._bad_paths:
                    self._bad_paths.add(img_path)
                    if len(self._bad_paths) <= 20:
                        print(f'WARNING: corrupt render #{len(self._bad_paths)}: {img_path}')
                    elif len(self._bad_paths) == 21:
                        print('WARNING: suppressing further corrupt image warnings...')
                idx = np.random.randint(len(self))
        raise RuntimeError(f'Failed after 10 attempts, {len(self._bad_paths)} corrupt images')

    def _load_sample(self, idx):
        uid_str, img_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        pixel_values = self.transform(img)
        anchor = self.brep_anchors[uid_str]
        return {
            'pixel_values': pixel_values,
            'anchor': anchor,
            'uid': uid_str,
        }

print('Dataset and transforms defined.')

In [ ]:
# Cell 8: Loss Functions
# Identical to sketch encoder: cosine pull + InfoNCE push at Matryoshka scales

def anchor_alignment_loss(z_pred, z_anchor, temperature=0.05):
    """Combined cosine + contrastive loss for aligning to anchor space.
    Pure MSE causes mode collapse (predict the mean).
    Pure InfoNCE wastes the anchor's geometric structure.
    Combined gives both metric matching and discriminative power.
    """
    z_pred_n = F.normalize(z_pred, dim=-1)
    z_anchor_n = F.normalize(z_anchor, dim=-1)

    # Cosine alignment (positive pair pull)
    cos_loss = 1.0 - (z_pred_n * z_anchor_n).sum(dim=-1).mean()

    # InfoNCE against in-batch anchors (negative pair push)
    B = z_pred.shape[0]
    logits = torch.clamp(z_pred_n @ z_anchor_n.T / temperature, -100, 100)
    labels = torch.arange(B, device=z_pred.device)
    contrastive_loss = F.cross_entropy(logits, labels)

    return cos_loss + contrastive_loss


def matryoshka_anchor_loss(z_pred, z_anchor, dims, weights, temperature=0.05):
    """Matryoshka-structured anchor alignment across multiple scales.
    Preserves the coarse-to-fine hierarchy of the BRep anchor space.
    """
    total = 0.0
    info = {}
    for dim, w in zip(dims, weights):
        loss = anchor_alignment_loss(z_pred[:, :dim], z_anchor[:, :dim], temperature)
        total += w * loss
        info[f'loss_{dim}d'] = loss.item()
    return total / sum(weights), info

print('Loss functions defined.')
print(f'Temperature: {config.TEMPERATURE} (fixed, matching v4 final tau)')

In [ ]:
# Cell 9: Training & Evaluation Functions

def train_one_epoch(model, loader, optimizer, scheduler, config):
    model.train()
    total_loss = 0.0
    n_batches = 0

    pbar = tqdm(loader, desc='Train')
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        anchors = batch['anchor'].to(device)

        z_pred = model(pixel_values)
        loss, info = matryoshka_anchor_loss(
            z_pred, anchors,
            config.MATRYOSHKA_DIMS, config.MATRYOSHKA_WEIGHTS,
            config.TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
        optimizer.step()
        if scheduler:
            scheduler.step()

        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            f'l_{config.D_SHARED}d': f'{info.get(f"loss_{config.D_SHARED}d", 0):.4f}'
        })

    return total_loss / max(n_batches, 1)


@torch.no_grad()
def evaluate(model, loader, config):
    model.eval()
    total_loss = 0.0
    all_z, all_uids = [], []
    n_batches = 0

    for batch in tqdm(loader, desc='Eval'):
        pixel_values = batch['pixel_values'].to(device)
        anchors = batch['anchor'].to(device)

        z_pred = model(pixel_values)
        loss, _ = matryoshka_anchor_loss(
            z_pred, anchors,
            config.MATRYOSHKA_DIMS, config.MATRYOSHKA_WEIGHTS,
            config.TEMPERATURE)

        total_loss += loss.item()
        n_batches += 1
        all_z.append(z_pred.cpu())
        all_uids.extend(batch['uid'])

    return total_loss / max(n_batches, 1), torch.cat(all_z), all_uids


@torch.no_grad()
def retrieval_eval(z_query, z_gallery, k_values=[1, 5, 10]):
    """Compute R@k: for each query i, check if gallery[i] is in top-k."""
    z_q = F.normalize(z_query, dim=-1)
    z_g = F.normalize(z_gallery, dim=-1)
    sim = z_q @ z_g.T  # (N, N)
    N = sim.shape[0]

    results = {}
    for k in k_values:
        topk_idx = sim.topk(k, dim=1).indices  # (N, k)
        correct = sum(1 for i in range(N) if i in topk_idx[i])
        results[f'R@{k}'] = 100.0 * correct / N
    return results


def full_eval(model, val_loader, brep_anchors_dict, config):
    """Full evaluation: render->brep retrieval at all Matryoshka dims."""
    _, z_render_all, uids_all = evaluate(model, val_loader, config)

    # Stack anchors in same order as render predictions
    z_brep = torch.stack([brep_anchors_dict[u] for u in uids_all])

    results = {}
    for dim in config.MATRYOSHKA_DIMS:
        z_r = z_render_all[:, :dim]
        z_b = z_brep[:, :dim]

        rb = retrieval_eval(z_r, z_b, config.K_VALUES)
        for k, v in rb.items():
            results[f'render2brep_{k}_{dim}d'] = v

    return results

print('Training and evaluation functions defined.')

In [ ]:
# Cell 10: Load Anchors + Create Datasets & DataLoaders

train_uids = [str(u).strip() for u in split_info['train_uids']]
val_uids = [str(u).strip() for u in split_info['val_uids']]
print(f'Split: {len(train_uids):,} train, {len(val_uids):,} val UIDs')

# --- Create datasets ---
train_dataset = RenderAnchorDataset(
    train_uids, RENDER_ROOT, brep_anchors, train_transform)
val_dataset = RenderAnchorDataset(
    val_uids, RENDER_ROOT, brep_anchors, val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
    num_workers=config.NUM_WORKERS, pin_memory=True,
    persistent_workers=True, drop_last=True)
val_loader = DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, pin_memory=True,
    persistent_workers=True)

print(f'\nTrain: {len(train_dataset):,} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset):,} samples, {len(val_loader)} batches')

In [ ]:
# Cell 11: Create SigLIP Model + Optimizer + Scheduler

render_model = SigLIPRenderEncoder(
    model_name=config.SIGLIP_MODEL, d_out=config.D_SHARED
).to(device)

# Differential LR: lower for pretrained SigLIP backbone, higher for projection head
optimizer = torch.optim.AdamW([
    {'params': render_model.vision.parameters(), 'lr': config.BACKBONE_LR,
     'weight_decay': config.WEIGHT_DECAY},
    {'params': render_model.projection.parameters(), 'lr': config.PROJ_LR,
     'weight_decay': config.WEIGHT_DECAY},
])

total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = len(train_loader) * config.WARMUP_EPOCHS


def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

n_params = sum(p.numel() for p in render_model.parameters()) / 1e6
n_trainable = sum(p.numel() for p in render_model.parameters() if p.requires_grad) / 1e6
print(f'SigLIP Render Encoder: {n_params:.1f}M total, {n_trainable:.1f}M trainable')
print(f'Optimizer: AdamW, backbone_lr={config.BACKBONE_LR:.0e}, proj_lr={config.PROJ_LR:.0e}')
print(f'Scheduler: cosine with {config.WARMUP_EPOCHS} warmup epochs')
print(f'Total steps: {total_steps}, warmup: {warmup_steps}')

In [ ]:
# Cell 12: Training Loop (with resume from checkpoint)

# === Resume logic ===
start_epoch = 0
best_r1 = 0.0
history = []

# Check Drive then local for existing checkpoint
resume_ckpt = None
for ckpt_dir in [config.SAVE_DIR, config.LOCAL_SAVE_DIR]:
    for fname in ['latest.pth', 'best.pth']:
        p = os.path.join(ckpt_dir, fname)
        if os.path.exists(p):
            resume_ckpt = p
            break
    if resume_ckpt:
        break

if resume_ckpt:
    ckpt = torch.load(resume_ckpt, map_location=device, weights_only=False)
    render_model.load_state_dict(ckpt['model_state_dict'])
    start_epoch = ckpt['epoch'] + 1

    if 'metrics' in ckpt:
        best_r1 = ckpt['metrics'].get(f'render2brep_R@1_{config.D_SHARED}d', 0)

    best_ckpt_path = os.path.join(os.path.dirname(resume_ckpt), 'best.pth')
    if os.path.exists(best_ckpt_path):
        best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        if 'optimizer_state_dict' in best_ckpt:
            try:
                optimizer.load_state_dict(best_ckpt['optimizer_state_dict'])
                print('Restored optimizer state from best.pth')
            except Exception as e:
                print(f'Could not restore optimizer: {e}')
        if 'metrics' in best_ckpt:
            best_r1 = max(best_r1, best_ckpt['metrics'].get(
                f'render2brep_R@1_{config.D_SHARED}d', 0))
        del best_ckpt

    # Fast-forward scheduler
    steps_done = start_epoch * len(train_loader)
    for _ in range(steps_done):
        scheduler.step()

    del ckpt
    torch.cuda.empty_cache()
    print(f'Resumed from epoch {start_epoch} (checkpoint: {resume_ckpt})')
    print(f'Best render2brep R@1 so far: {best_r1:.2f}%')
    print(f'Remaining epochs: {start_epoch+1}-{config.NUM_EPOCHS}')
else:
    print('No checkpoint found. Starting from scratch.')

# === Training loop ===
for epoch in range(start_epoch, config.NUM_EPOCHS):
    t0 = time.time()
    print(f'\nEpoch {epoch + 1}/{config.NUM_EPOCHS}')
    print('-' * 50)

    # Train
    train_loss = train_one_epoch(
        render_model, train_loader, optimizer, scheduler, config)

    # Evaluate
    val_loss, _, _ = evaluate(render_model, val_loader, config)

    # Retrieval metrics
    metrics = full_eval(render_model, val_loader, brep_anchors, config)

    # Print results
    lr_bb = optimizer.param_groups[0]['lr']
    lr_proj = optimizer.param_groups[1]['lr']
    elapsed = time.time() - t0
    print(f'  Train loss: {train_loss:.4f}  Val loss: {val_loss:.4f}  '
          f'LR: bb={lr_bb:.2e} proj={lr_proj:.2e}  Time: {elapsed:.0f}s')

    r1 = metrics.get(f'render2brep_R@1_{config.D_SHARED}d', 0)
    r5 = metrics.get(f'render2brep_R@5_{config.D_SHARED}d', 0)
    r10 = metrics.get(f'render2brep_R@10_{config.D_SHARED}d', 0)
    print(f'  render2brep @ {config.D_SHARED}d: R@1={r1:.2f}% R@5={r5:.2f}% R@10={r10:.2f}%')

    history.append({
        'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss,
        **metrics, 'lr_backbone': lr_bb, 'lr_proj': lr_proj,
    })

    # Save checkpoint
    is_best = r1 > best_r1
    if is_best:
        best_r1 = r1

    ckpt = {
        'epoch': epoch,
        'model_state_dict': render_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'config': {k: v for k, v in vars(Config).items()
                   if not k.startswith('_')},
    }

    # Save to both local + Drive
    for ckpt_dir in [config.LOCAL_SAVE_DIR, config.SAVE_DIR]:
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save(ckpt, os.path.join(ckpt_dir, 'latest.pth'))
        if is_best:
            torch.save(ckpt, os.path.join(ckpt_dir, 'best.pth'))
        if (epoch + 1) % config.SAVE_EVERY == 0:
            torch.save(ckpt, os.path.join(ckpt_dir, f'epoch_{epoch+1}.pth'))

    if is_best:
        print(f'  *** New best R@1: {best_r1:.2f}% ***')

    del ckpt
    torch.cuda.empty_cache()

print(f'\nTraining complete. Best render2brep R@1: {best_r1:.2f}%')

In [ ]:
# Cell 13: Final Evaluation - Full Matryoshka Matrix

print('=== Final Evaluation ===')
print()

# Load best checkpoint
best_path = os.path.join(config.LOCAL_SAVE_DIR, 'best.pth')
if not os.path.exists(best_path):
    best_path = os.path.join(config.SAVE_DIR, 'best.pth')

if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    render_model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best checkpoint from epoch {ckpt["epoch"]+1}')
    del ckpt
    torch.cuda.empty_cache()
else:
    print('Using current model (no best checkpoint found)')

metrics = full_eval(render_model, val_loader, brep_anchors, config)

print(f'\n{"Dim":>6} | {"render->brep R@1":>17} {"R@5":>6} {"R@10":>6}')
print('-' * 50)
for dim in config.MATRYOSHKA_DIMS:
    rb1 = metrics.get(f'render2brep_R@1_{dim}d', 0)
    rb5 = metrics.get(f'render2brep_R@5_{dim}d', 0)
    rb10 = metrics.get(f'render2brep_R@10_{dim}d', 0)
    print(f'{dim:>6}d | {rb1:>16.2f}% {rb5:>5.2f}% {rb10:>5.2f}%')

# Print training history summary
print(f'\n=== Training History ===')
for h in history:
    rb = h.get(f'render2brep_R@1_{config.D_SHARED}d', 0)
    print(f'  Epoch {h["epoch"]:2d}: train={h["train_loss"]:.4f} val={h["val_loss"]:.4f} '
          f'render2brep_R@1={rb:.2f}%')

In [ ]:
# Cell 14: Training Curves

if history:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = [h['epoch'] for h in history]

    ax1.plot(epochs, [h['train_loss'] for h in history],
             'b-o', label='Train', markersize=4)
    ax1.plot(epochs, [h['val_loss'] for h in history],
             'r-o', label='Val', markersize=4)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    for dim in config.MATRYOSHKA_DIMS:
        key = f'render2brep_R@1_{dim}d'
        vals = [h.get(key, 0) for h in history]
        ax2.plot(epochs, vals, '-o', label=f'{dim}d', markersize=4)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('R@1 (%)')
    ax2.set_title('Render -> BRep R@1 by Matryoshka Dim')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(config.SAVE_DIR, 'training_curves.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved training curves to {plot_path}')
else:
    print('No training history to plot.')

In [ ]:
# Cell 15: Export Render Embeddings for Inference Pipeline
# Computes render embeddings for ALL UIDs (train+val) using best model,
# saves as render_embeddings.pt for the unified inference notebook.

print('Computing render embeddings for all UIDs...')

# Load best model
best_path = os.path.join(config.LOCAL_SAVE_DIR, 'best.pth')
if not os.path.exists(best_path):
    best_path = os.path.join(config.SAVE_DIR, 'best.pth')
ckpt = torch.load(best_path, map_location=device, weights_only=False)
render_model.load_state_dict(ckpt['model_state_dict'])
render_model.eval()
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]+1})')
del ckpt
torch.cuda.empty_cache()

# Full dataset (no augmentation, val transform)
all_uids = train_uids + val_uids
full_dataset = RenderAnchorDataset(
    all_uids, RENDER_ROOT, brep_anchors, val_transform)
full_loader = DataLoader(
    full_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=config.NUM_WORKERS, pin_memory=True)

all_z = []
all_uid_list = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc='Encoding renders'):
        pixel_values = batch['pixel_values'].to(device)
        z = render_model(pixel_values).cpu()
        all_z.append(z)
        all_uid_list.extend(batch['uid'])

all_z = torch.cat(all_z, dim=0)

# Save as dict: uid -> embedding
render_embeddings = {uid: all_z[i] for i, uid in enumerate(all_uid_list)}

save_path = os.path.join(config.SAVE_DIR, 'render_embeddings.pt')
torch.save(render_embeddings, save_path)
print(f'\nSaved {len(render_embeddings):,} render embeddings to {save_path}')
print(f'Embedding shape: {all_z.shape}')

# Also save locally if dirs differ
if config.SAVE_DIR != config.LOCAL_SAVE_DIR:
    import shutil
    local_path = os.path.join(config.LOCAL_SAVE_DIR, 'render_embeddings.pt')
    shutil.copy2(save_path, local_path)
    print(f'Copied to local: {local_path}')

torch.cuda.empty_cache()
gc.collect()
print('Done!')